<a href="https://colab.research.google.com/github/TowsifTazwar/data-science-research/blob/main/6_Data_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler, RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
import warnings; warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='Set2')

In [8]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
df = pd.read_csv('/content/drive/MyDrive/Datasets/titanic_train.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [11]:


data = pd.read_csv('/content/drive/MyDrive/Datasets/titanic_train.csv')

#Inject NaNs

np.random.seed(42)

for col, n in {
    'Age': 10,
    'Fare': 5,
    'Embarked': 3
}.items():
    data.loc[np.random.choice(data.index, n, replace=False), col] = np.nan


#Fill Missing Values

data['Age'].fillna(data['Age'].median(), inplace=True)
data['Fare'].fillna(data['Fare'].mean(), inplace=True)
data['Embarked'].fillna(data['Embarked'].mode()[0], inplace=True)


data.drop(columns=['Cabin'], inplace=True)


#Clean Data

data.drop_duplicates(inplace=True)

#Dropping unnecessary columns
data.drop(columns=['PassengerId', 'Name', 'Ticket'], inplace=True)


#Outlier Removal

for col in ['Age', 'Fare']:
    q1, q3 = data[col].quantile([0.25, 0.75])
    iqr = q3 - q1

    data = data[
        data[col].between(
            q1 - 1.5 * iqr,
            q3 + 1.5 * iqr
        )
    ]


#Encoding

data = pd.get_dummies(
    data,
    columns=['Sex', 'Embarked'],
    drop_first=True
)


#Split Features & Target

X = data.drop(columns=['Survived']).values
y = data['Survived'].values

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


#Scaling

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Pipeline complete → X_train: {X_train.shape} | X_test: {X_test.shape}")

Pipeline complete → X_train: (576, 8) | X_test: (144, 8)
